In [ ]:
import numpy as np

In [ ]:
class Affine:
    def __init__(self, weight, bias):
        self.weight = weight
        self.bias = bias

    def forward(self, x):
        return x @ self.weight.T + self.bias

    def __call__(self, x):
        return self.forward(x)

In [ ]:
class Identity:
  def __init__(self):
    pass

  def forward(self, x):
    return x

  def __call__(self, x):
    return self.forward(x)

In [ ]:
class Threshold:
    def __init__(self):
        pass

    def forward(self, x):
        return (x >= 0).astype(int)

    def __call__(self, x):
        return self.forward(x)

In [ ]:
class Sigmoid:
  def __init__(self):
    pass

  def forward(self, x):
    return 1/(1 + np.exp(-x))

  def __call__(self, x):
    return self.forward(x)

In [ ]:
class Tanh:
  def __init__(self):
    pass

  def forward(self, x):
    return (np.exp(x) - np.exp(-x))/(np.exp(x) + np.exp(-x))

  def __call__(self, x):
    return self.forward(x)

In [ ]:
class ReLU:
  def __init__(self):
    pass

  def forward(self, x):
    return np.maximum(0, x)

  def __call__(self, x):
    return self.forward(x)

In [ ]:
class GELU:
   def __init__(self):
     pass

   def forward(self, x):
     return x * 0.5 * (1 + np.erf(x / np.sqrt(2))) # erf ist the ge function

   def __call__(self, x):
     return self.forward(x)

In [ ]:
class MLP_AND:
    def __init__(self):
        self.affine1 = Affine(np.array([[1, 1]]), np.array([-2])) # (2, ) <- is vector vs (2, 1) <- is matrix for [1, 1] and [[1, 1]]
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]
    def forward(self, x):

        return self.activation1(self.affine1(x)).astype(bool)

    def __call__(self, x):
        return self.forward(x)

In [ ]:
class MLP_OR:
    def __init__(self):
        self.affine1 = Affine(np.array([[1, 1]]), np.array([-1])) # (2, ) <- is vector vs (2, 1) <- is matrix for [1, 1] and [[1, 1]]
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]
    def forward(self, x):

        return self.activation1(self.affine1(x)).astype(bool)

    def __call__(self, x):
        return self.forward(x)

In [ ]:
class MLP_NOT:
    def __init__(self):
        self.affine1 = Affine(np.array([[-1]]), np.array([1])) # (2, ) <- is vector vs (2, 1) <- is matrix for [1, 1] and [[1, 1]]
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]
    def forward(self, x):

        return self.activation1(self.affine1(x)).astype(bool)

    def __call__(self, x):
        return self.forward(x)

In [ ]:
class MLP_Upper:
    def __init__(self):
        self.affine1 = Affine(np.array([[1/np.sqrt(2), 1/np.sqrt(2)]]), np.array([-1/np.sqrt(2)]))
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]

    def forward(self, x):
        for module in self.layer1:
                x = module(x)

        return x.astype(bool)
    def __call__(self, x):
        return self.forward(x)

In [ ]:
class MLP_Triangle:
    def __init__(self):
        self.affine1 = Affine(np.array([[-1, 0],[0, -1],[1/np.sqrt(2), 1/np.sqrt(2)]]), np.array([0,0, -1/np.sqrt(2)]))
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]

        self.affine2 = Affine(np.array([[-1, -1, -1]]), np.array([0]))
        self.activation2 = Threshold()
        self.layer2 = [self.affine2, self.activation2]
        self.layer = [self.layer1, self.layer2]
    def forward(self, x):
        for layer in self.layer:
            for module in layer:
                x = module(x)

        return x.astype(bool)
    def __call__(self, x):
        return self.forward(x)

In [ ]:
class MLP_Krange:
    def __init__(self, Krange, Hvals):

        weight = np.tile([1, -1], (len(Krange) - 1, 1))
        bias  = []
        for i in range(len(Krange) - 1):
            bias.append([-Krange[i], Krange[i + 1]])
        bias = np.array(bias)
        self.affine1 = Affine(weight, bias.T)
        self.activation1 = Threshold()
        self.layer1 = [self.affine1, self.activation1]

        self.affine2 = Affine(np.array([[1, 1]]), np.array([[-2]]))
        self.activation2 = Threshold()
        self.layer2 = [self.affine2, self.activation2]

        self.affine3 = Affine(np.array([Hvals]), np.array([0]))
        self.activation3 = Identity()
        self.layer3 = [self.affine3, self.activation3]
        self.layer = [self.layer1, self.layer2, self.layer3]



    def forward(self, z : np.ndarray):
        ret = []
        for y, x in enumerate(z):
            x = np.array([[x, 0], [0, x]])
            x = self.activation1(self.affine1(x)).T
            x = self.activation2(self.affine2(x)).T
            x = self.activation3(self.affine3(x))
            ret.append(x.astype(int))

        ret = np.array(ret)
        return ret
    def __call__(self, x):
        return self.forward(x)

if you are doing binary classification then your one bit represents you confidnece that is is the first case if it is close to zero then you are confident it is not that and therefore you ae cofidnent that it is the other option if you are close to one then it is self explanitory

you never apply a activatiokn function to the last layer

In [ ]:
Krange = np.array([-1, 1 , 3 ,5, 7, 9, 11, 13])
Hval = np.array([1, 2, 3, 4, 5, 6 , 7])

Krangecheck = MLP_Krange(Krange, Hval)
n_samples = 10
np.random.seed(42)
x = np.random.uniform(0, 10, size=n_samples)
np.random.seed()
y = Krangecheck(x)
y = np.array(y)
print('x', x, '\n')
print('shape:', x.shape, '\n')


print('x', y, '\n')
print('shape:', y.shape, '\n')

x [3.74540119 9.50714306 7.31993942 5.98658484 1.5601864  1.5599452
 0.58083612 8.66176146 6.01115012 7.08072578] 

shape: (10,) 

x [[[3]]

 [[6]]

 [[5]]

 [[4]]

 [[2]]

 [[2]]

 [[1]]

 [[5]]

 [[4]]

 [[5]]] 

shape: (10, 1, 1) 



because we generally care about what is the largest value and using an activation function will generally not change that.But some time you will need to clip the large values.

for binary classificatikn you only need one logit becase if you use a logit then if hte output is grater then 0 then it is the fist calss other wise it is the second class tecnicly ofr milit class you only need n-1 but that is an unnesisary optimizatinon so it is not done.

A is the last layer is unbonded then you could use a liner activaation.

B I would do max [a, x] to make the values in the given range

c I would do min [a, x] to make the values in the given range

D I would do logistic and thenmutiply by b - a and then add a as that would put the value in the target range

